In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('silver_daily_ohlcv_2000_2025.csv')
print(df.shape)
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'silver_daily_ohlcv_2000_2025.csv'

In [ ]:
print(df.describe())

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(df['Date'], df['Close'])
plt.title('Silver Price 2000-2025')
plt.xlabel('Year')
plt.ylabel('Price (USD)')
plt.show()

In [ ]:
print(df.isnull().sum())

In [ ]:
Q1 = df['Close'].quantile(0.25)
Q3 = df['Close'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[df['Close'] > Q3 + 4*IQR]
print(f"Outliers found: {len(outliers)}")
print(outliers[['Date','Close']])

In [ ]:
import seaborn as sns

numeric_cols = df[['Open','High','Low','Close','Volume','Returns_Pct']]
plt.figure(figsize=(10,6))
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(df['Close'])
print(f'ADF Statistic: {result[0]}')
print(f'p-value: {result[1]}')
if result[1] < 0.05:
    print("Data is Stationary")
else:
    print("Data is Non-Stationary")

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1,2, figsize=(14,4))
plot_acf(df['Close'], lags=40, ax=axes[0])
plot_pacf(df['Close'], lags=40, ax=axes[1])
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1,2, figsize=(14,4))
plot_acf(df['Log_Returns'].dropna(), lags=40, ax=axes[0])
plot_pacf(df['Log_Returns'].dropna(), lags=40, ax=axes[1])
plt.title('ACF/PACF for Log Returns')
plt.show()


In [ ]:
from statsmodels.tsa.stattools import kpss

result = kpss(df['Close'])
print(f'KPSS Statistic: {result[0]}')
print(f'p-value: {result[1]}')
if result[1] < 0.05:
    print("Data is Non-Stationary")
else:
    print("Data is Stationary")

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date').reset_index(drop=True)
print("Done!")
print(df.dtypes)

In [ ]:
# Split data: 70% train, 15% validation, 15% test
train_end = '2017-12-31'
val_end = '2020-12-31'

train = df[df['Date'] <= train_end]
val = df[(df['Date'] > train_end) & (df['Date'] <= val_end)]
test = df[df['Date'] > val_end]

print(f"Train: {len(train)} rows (2000-2017)")
print(f"Validation: {len(val)} rows (2018-2020)")
print(f"Test: {len(test)} rows (2021-2025)")


In [ ]:

from sklearn.preprocessing import MinMaxScaler, RobustScaler

# MinMaxScaler
minmax = MinMaxScaler()
train_minmax = minmax.fit_transform(train[['Close']])
val_minmax = minmax.transform(val[['Close']])
test_minmax = minmax.transform(test[['Close']])

# RobustScaler
robust = RobustScaler()
train_robust = robust.fit_transform(train[['Close']])
val_robust = robust.transform(val[['Close']])
test_robust = robust.transform(test[['Close']])

print("Scaling done!")
print(f"MinMax range: {train_minmax.min():.3f} to {train_minmax.max():.3f}")
print(f"Robust range: {train_robust.min():.3f} to {train_robust.max():.3f}")


In [ ]:
import pickle

# Save splits
with open('train.pkl', 'wb') as f:
    pickle.dump(train, f)
with open('val.pkl', 'wb') as f:
    pickle.dump(val, f)
with open('test.pkl', 'wb') as f:
    pickle.dump(test, f)

print("Data saved successfully!")

In [ ]:
import numpy as np

def create_sequences(data, lookback=60, horizon=1):
    X, y = [], []
    for i in range(lookback, len(data) - horizon + 1):
        X.append(data[i-lookback:i])
        y.append(data[i:i+horizon])
    return np.array(X), np.array(y)

# Create sequences
X_train, y_train = create_sequences(train_minmax, lookback=60, horizon=1)
X_val, y_val = create_sequences(val_minmax, lookback=60, horizon=1)
X_test, y_test = create_sequences(test_minmax, lookback=60, horizon=1)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")


In [ ]:
# Load all data files
df_futures = pd.read_csv('silver_futures_contracts.csv')
df_sentiment = pd.read_csv('silver_sentiment_weekly.csv')
df_supply = pd.read_csv('silver_supply_demand_annual.csv')
df_macro = pd.read_csv('silver_macroeconomic_monthly.csv')

print("Futures:", df_futures.shape)
print("Sentiment:", df_sentiment.shape)
print("Supply:", df_supply.shape)
print("Macro:", df_macro.shape)

In [ ]:
# Convert dates
df_futures['Date'] = pd.to_datetime(df_futures['Date'], dayfirst=True)
df_sentiment['Date'] = pd.to_datetime(df_sentiment['Date'], dayfirst=True)
df_macro['Date'] = pd.to_datetime(df_macro['Date'], dayfirst=True)
df_supply['Year'] = pd.to_datetime(df_supply['Year'], format='%Y')

# Merge main data with futures
df_merged = pd.merge(df, df_futures[['Date']], on='Date', how='left')
print(f"After merge: {df_merged.shape}")
print("Done!")

In [ ]:
print(df_futures.columns.tolist())

In [ ]:
# Convert dates with correct column names
df_futures['Trade_Date'] = pd.to_datetime(df_futures['Trade_Date'], dayfirst=True)
df_sentiment['Date'] = pd.to_datetime(df_sentiment['Date'], dayfirst=True)
df_macro['Date'] = pd.to_datetime(df_macro['Date'], dayfirst=True)
df_supply['Year'] = pd.to_datetime(df_supply['Year'], format='%Y')

# Merge
df_merged = pd.merge(df, df_futures[['Trade_Date', 'Futures_Price']],
                     left_on='Date', right_on='Trade_Date', how='left')
print(f"After merge: {df_merged.shape}")
print("Done!")

In [ ]:
print(df_futures['Trade_Date'].head())
print(df_sentiment['Date'].head())

In [ ]:
# Convert dates
df_futures['Date'] = pd.to_datetime(df_futures['Date'], dayfirst=True)
df_sentiment['Date'] = pd.to_datetime(df_sentiment['Date'], dayfirst=True)
df_macro['Date'] = pd.to_datetime(df_macro['Date'], dayfirst=True)
df_supply['Year'] = pd.to_datetime(df_supply['Year'], format='%Y')

# Merge main data with futures
df_merged = pd.merge(df, df_futures[['Date']], on='Date', how='left')
print(f"After merge: {df_merged.shape}")
print("Done!")

In [ ]:
df_futures['Trade_Date'] = pd.to_datetime(df_futures['Trade_Date'], dayfirst=True)
df_sentiment['Date'] = pd.to_datetime(df_sentiment['Date'], dayfirst=True)
df_macro['Date'] = pd.to_datetime(df_macro['Date'], dayfirst=True)
df_supply['Year'] = pd.to_datetime(df_supply['Year'], format='%Y')

# Merge
df_merged = pd.merge(df, df_futures[['Trade_Date', 'Futures_Price']],
                     left_on='Date', right_on='Trade_Date', how='left')
print(f"After merge: {df_merged.shape}")
print("Done!")

In [ ]:
print(df_futures['Trade_Date'].head())
print(df_sentiment['Date'].head())

In [ ]:
print("Futures columns:", df_futures.columns.tolist())
print("Sentiment columns:", df_sentiment.columns.tolist())
print("Supply columns:", df_supply.columns.tolist())
print("Macro columns:", df_macro.columns.tolist())

In [ ]:
# Fix date columns
df_futures['Trade_Date'] = pd.to_datetime(df_futures['Trade_Date'])
df_sentiment['Week_Ending'] = pd.to_datetime(df_sentiment['Week_Ending'])
df_supply['Year'] = pd.to_datetime(df_supply['Year'], format='%Y')
df_macro['Date'] = pd.to_datetime(df_macro['Date'])

# Merge with main df
df_merged = df.copy()

# Merge futures
df_merged = pd.merge(df_merged, df_futures[['Trade_Date','Futures_Price','Basis']],
                     left_on='Date', right_on='Trade_Date', how='left')

# Merge macro (monthly - forward fill)
df_merged = pd.merge(df_merged, df_macro[['Date','Fed_Funds_Rate','DXY_Index','Gold_Price_USD','VIX_Index']],
                     on='Date', how='left').ffill()

print(f"Merged shape: {df_merged.shape}")
print("Done!")


In [ ]:
# Fix date columns
df_futures['Trade_Date'] = pd.to_datetime(df_futures['Trade_Date'])
df_sentiment['Week_Ending'] = pd.to_datetime(df_sentiment['Week_Ending'], errors='coerce')
df_supply['Year'] = pd.to_datetime(df_supply['Year'], format='%Y')
df_macro['Date'] = pd.to_datetime(df_macro['Date'])

# Merge with main df
df_merged = df.copy()

# Merge futures
df_merged = pd.merge(df_merged, df_futures[['Trade_Date','Futures_Price','Basis']],
                     left_on='Date', right_on='Trade_Date', how='left')

# Merge macro
df_merged = pd.merge(df_merged, df_macro[['Date','Fed_Funds_Rate','DXY_Index','Gold_Price_USD','VIX_Index']],
                     on='Date', how='left').ffill()

print(f"Merged shape: {df_merged.shape}")
print("Done!")


In [ ]:
df_merged.to_csv('silver_merged.csv', index=False)
print("Merged data saved!")


In [ ]:
# Merge sentiment (weekly - forward fill)
df_sentiment['Week_Ending'] = pd.to_datetime(df_sentiment['Week_Ending'], errors='coerce')

df_merged2 = pd.merge_asof(df_merged.sort_values('Date'),
                            df_sentiment[['Week_Ending','News_Sentiment_Score','Google_Trends_Index','Implied_Volatility_30D']].sort_values('Week_Ending'),
                            left_on='Date',
                            right_on='Week_Ending',
                            direction='backward')

print(f"Shape after sentiment merge: {df_merged2.shape}")
print("Done!")

In [ ]:
# Drop null dates from sentiment
df_sentiment_clean = df_sentiment.dropna(subset=['Week_Ending'])
df_sentiment_clean['Week_Ending'] = pd.to_datetime(df_sentiment_clean['Week_Ending'], errors='coerce')
df_sentiment_clean = df_sentiment_clean.dropna(subset=['Week_Ending']).sort_values('Week_Ending')

df_merged2 = pd.merge_asof(df_merged.sort_values('Date'),
                            df_sentiment_clean[['Week_Ending','News_Sentiment_Score','Google_Trends_Index','Implied_Volatility_30D']].sort_values('Week_Ending'),
                            left_on='Date',
                            right_on='Week_Ending',
                            direction='backward')

print(f"Shape: {df_merged2.shape}")
print("Done!")

In [ ]:
# Create lag features for XGBoost
df_merged2 = df_merged2.copy()

for lag in [1, 2, 3, 5, 10, 20]:
    df_merged2[f'Close_lag_{lag}'] = df_merged2['Close'].shift(lag)

print("Lag features created!")
print(df_merged2[['Date','Close','Close_lag_1','Close_lag_5','Close_lag_20']].head(25))


In [ ]:
# SMA - Simple Moving Average
df_merged2['SMA_20'] = df_merged2['Close'].rolling(window=20).mean()
df_merged2['SMA_50'] = df_merged2['Close'].rolling(window=50).mean()

# EMA - Exponential Moving Average
df_merged2['EMA_20'] = df_merged2['Close'].ewm(span=20).mean()
df_merged2['EMA_50'] = df_merged2['Close'].ewm(span=50).mean()

print("SMA and EMA done!")
print(df_merged2[['Date','Close','SMA_20','EMA_20']].tail())


In [ ]:
# MACD
exp1 = df_merged2['Close'].ewm(span=12).mean()
exp2 = df_merged2['Close'].ewm(span=26).mean()
df_merged2['MACD'] = exp1 - exp2
df_merged2['MACD_Signal'] = df_merged2['MACD'].ewm(span=9).mean()
df_merged2['MACD_Histogram'] = df_merged2['MACD'] - df_merged2['MACD_Signal']

print("MACD done!")


In [ ]:
# RSI - Relative Strength Index
delta = df_merged2['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df_merged2['RSI'] = 100 - (100 / (1 + rs))

print("RSI done!")
print(df_merged2['RSI'].tail())

In [ ]:
# Bollinger Bands
df_merged2['BB_Middle'] = df_merged2['Close'].rolling(window=20).mean()
df_merged2['BB_Upper'] = df_merged2['BB_Middle'] + 2*df_merged2['Close'].rolling(window=20).std()
df_merged2['BB_Lower'] = df_merged2['BB_Middle'] - 2*df_merged2['Close'].rolling(window=20).std()
df_merged2['BB_Width'] = df_merged2['BB_Upper'] - df_merged2['BB_Lower']

print("Bollinger Bands done!")

In [ ]:
# ATR - Average True Range
df_merged2['TR'] = np.maximum(df_merged2['High'] - df_merged2['Low'],
                   np.maximum(abs(df_merged2['High'] - df_merged2['Close'].shift(1)),
                              abs(df_merged2['Low'] - df_merged2['Close'].shift(1))))
df_merged2['ATR'] = df_merged2['TR'].rolling(window=14).mean()

print("ATR done!")

In [ ]:
# Stochastic Oscillator
low_14 = df_merged2['Low'].rolling(window=14).min()
high_14 = df_merged2['High'].rolling(window=14).max()
df_merged2['Stoch_K'] = 100 * (df_merged2['Close'] - low_14) / (high_14 - low_14)
df_merged2['Stoch_D'] = df_merged2['Stoch_K'].rolling(window=3).mean()

print("Stochastic done!")

In [ ]:
    np.maximum(df_merged2['High'] - df_merged2['High'].shift(1), 0), 0)

df_merged2['DM_Minus'] = np.where(
    (df_merged2['Low'].shift(1) - df_merged2['Low']) > (df_merged2['High'] - df_merged2['High'].shift(1)),
    np.maximum(df_merged2['Low'].shift(1) - df_merged2['Low'], 0), 0)

df_merged2['DI_Plus'] = 100 * (df_merged2['DM_Plus'].rolling(14).mean() / df_merged2['ATR'])
df_merged2['DI_Minus'] = 100 * (df_merged2['DM_Minus'].rolling(14).mean() / df_merged2['ATR'])
df_merged2['ADX'] = 100 * abs(df_merged2['DI_Plus'] - df_merged2['DI_Minus']) / (df_merged2['DI_Plus'] + df_merged2['DI_Minus'])
df_merged2['ADX'] = df_merged2['ADX'].rolling(14).mean()

print("ADX done!")

In [ ]:
# ADX - Average Directional Index
df_merged2['DM_Plus'] = np.where(
    (df_merged2['High'] - df_merged2['High'].shift(1)) > (df_merged2['Low'].shift(1) - df_merged2['Low']),
    np.maximum(df_merged2['High'] - df_merged2['High'].shift(1), 0), 0)

df_merged2['DM_Minus'] = np.where(
    (df_merged2['Low'].shift(1) - df_merged2['Low']) > (df_merged2['High'] - df_merged2['High'].shift(1)),
    np.maximum(df_merged2['Low'].shift(1) - df_merged2['Low'], 0), 0)

df_merged2['DI_Plus'] = 100 * (df_merged2['DM_Plus'].rolling(14).mean() / df_merged2['ATR'])
df_merged2['DI_Minus'] = 100 * (df_merged2['DM_Minus'].rolling(14).mean() / df_merged2['ATR'])
df_merged2['ADX'] = 100 * abs(df_merged2['DI_Plus'] - df_merged2['DI_Minus']) / (df_merged2['DI_Plus'] + df_merged2['DI_Minus'])
df_merged2['ADX'] = df_merged2['ADX'].rolling(14).mean()

print("ADX done!")


In [ ]:
# OBV - On Balance Volume
df_merged2['OBV'] = (np.sign(df_merged2['Close'].diff()) * df_merged2['Volume']).cumsum()

print("OBV done!")

In [ ]:
# VWAP - Volume Weighted Average Price
df_merged2['VWAP_Calc'] = (df_merged2['Close'] * df_merged2['Volume']).cumsum() / df_merged2['Volume'].cumsum()

print("VWAP done!")

In [ ]:
# Volatility Features
df_merged2['Volatility_5'] = df_merged2['Log_Returns'].rolling(window=5).std()
df_merged2['Volatility_10'] = df_merged2['Log_Returns'].rolling(window=10).std()
df_merged2['Volatility_20'] = df_merged2['Log_Returns'].rolling(window=20).std()
df_merged2['Volatility_60'] = df_merged2['Log_Returns'].rolling(window=60).std()

# Parkinson Volatility
df_merged2['Parkinson_Vol'] = np.sqrt(
    (1/(4*np.log(2))) * ((np.log(df_merged2['High']/df_merged2['Low']))**2).rolling(20).mean()
)

print("Volatility features done!")

In [ ]:
# Ichimoku
high_9 = df_merged2['High'].rolling(9).max()
low_9 = df_merged2['Low'].rolling(9).min()
high_26 = df_merged2['High'].rolling(26).max()
low_26 = df_merged2['Low'].rolling(26).min()
high_52 = df_merged2['High'].rolling(52).max()
low_52 = df_merged2['Low'].rolling(52).min()

df_merged2['Ichimoku_Tenkan'] = (high_9 + low_9) / 2
df_merged2['Ichimoku_Kijun'] = (high_26 + low_26) / 2
df_merged2['Ichimoku_SpanA'] = ((df_merged2['Ichimoku_Tenkan'] + df_merged2['Ichimoku_Kijun']) / 2).shift(26)
df_merged2['Ichimoku_SpanB'] = ((high_52 + low_52) / 2).shift(26)